# HEALTHCARE CLINIC OPTIMIZATION — STRATEGY IMPLEMENTATION

## Report 1: Business Case — AI-Powered No-Show Reduction & Revenue Recovery
## Client : Healthcare Clinic Chain (5 Clinics)
## Author : Strategy & Analytics Division
## Date   : May 2026

This module implements all four strategic interventions described in the report:

  Module 1 — Current State Analysis & Revenue Metrics

  Module 2 — No-Show Prediction Model (ML)

  Module 3 — Smart Overbooking Engine


  Module 4 — Patient Segmentation (RFM)

  Module 5 — Automated Reminder Scheduler
  
  Module 6 — KPI Dashboard & Visualizations



In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
import os

warnings.filterwarnings("ignore")
np.random.seed(42)


# GLOBAL CONFIGURATION


In [6]:
CLINICS         = 5
PATIENTS_PER_CLINIC_DAY = 40
NO_SHOW_RATE    = 0.30
REVENUE_PER_VISIT = 700       # INR
WORKING_DAYS    = 26          # per month
TOTAL_SLOTS_DAY = CLINICS * PATIENTS_PER_CLINIC_DAY  # 200

PALETTE = {
    "teal":   "#1D9E75",
    "red":    "#E24B4A",
    "blue":   "#378ADD",
    "amber":  "#EF9F27",
    "purple": "#7F77DD",
    "gray":   "#888780",
    "light":  "#F1EFE8",
    "dark":   "#2C2C2A",
}

OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# MODULE 1: CURRENT STATE ANALYSIS

In [7]:
class CurrentStateAnalysis:
    """
    Quantifies the current revenue position, lost revenue due to no-shows,
    and slot utilisation across the 5-clinic chain.
    """

    def __init__(self):
        self.total_slots_day   = TOTAL_SLOTS_DAY
        self.attended_day      = int(self.total_slots_day * (1 - NO_SHOW_RATE))
        self.no_show_slots_day = self.total_slots_day - self.attended_day
        self.actual_daily_rev  = self.attended_day * REVENUE_PER_VISIT
        self.actual_monthly_rev= self.actual_daily_rev * WORKING_DAYS
        self.max_daily_rev     = self.total_slots_day * REVENUE_PER_VISIT
        self.max_monthly_rev   = self.max_daily_rev * WORKING_DAYS
        self.lost_daily_rev    = self.no_show_slots_day * REVENUE_PER_VISIT
        self.lost_monthly_rev  = self.lost_daily_rev * WORKING_DAYS
        self.utilisation       = self.attended_day / self.total_slots_day

    def summary(self) -> pd.DataFrame:
        data = {
            "Metric": [
                "Total scheduled appointments/day",
                "Actual patients seen/day (70% utilisation)",
                "No-show slots/day",
                "Actual daily revenue (₹)",
                "Actual monthly revenue (₹)",
                "Theoretical max daily revenue (₹)",
                "Theoretical max monthly revenue (₹)",
                "Daily revenue LOST to no-shows (₹)",
                "Monthly revenue LOST to no-shows (₹)",
                "Current slot utilisation rate",
            ],
            "Value": [
                self.total_slots_day,
                self.attended_day,
                self.no_show_slots_day,
                f"₹{self.actual_daily_rev:,}",
                f"₹{self.actual_monthly_rev:,}",
                f"₹{self.max_daily_rev:,}",
                f"₹{self.max_monthly_rev:,}",
                f"₹{self.lost_daily_rev:,}",
                f"₹{self.lost_monthly_rev:,}",
                f"{self.utilisation:.0%}",
            ],
        }
        return pd.DataFrame(data)

    def revenue_at_target(self, target_no_show: float) -> dict:
        """Calculate recovered revenue at a given target no-show rate."""
        attended   = int(self.total_slots_day * (1 - target_no_show))
        monthly_rev= attended * REVENUE_PER_VISIT * WORKING_DAYS
        uplift     = monthly_rev - self.actual_monthly_rev
        return {
            "target_no_show": target_no_show,
            "attended_per_day": attended,
            "monthly_revenue": monthly_rev,
            "monthly_uplift": uplift,
            "utilisation": attended / self.total_slots_day,
        }

    def plot_revenue_gap(self, ax=None):
        """Bar chart: current vs scenarios vs max revenue."""
        scenarios = [0.30, 0.22, 0.14, 0.00]
        labels    = ["Current\n(30%)", "3-month\n(22%)", "12-month\n(14%)", "Max\n(0%)"]
        revenues  = [self.revenue_at_target(r)["monthly_revenue"] / 1e5 for r in scenarios]
        colors    = [PALETTE["gray"], PALETTE["blue"], PALETTE["teal"], PALETTE["teal"]]
        alphas    = [1.0, 0.7, 1.0, 0.5]

        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 5))

        bars = ax.bar(labels, revenues, color=colors, alpha=0.9, width=0.55, zorder=3)
        for bar, rev, alpha in zip(bars, revenues, alphas):
            bar.set_alpha(alpha)
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f"₹{rev:.1f}L",
                ha="center", va="bottom", fontsize=11, fontweight="bold",
                color=PALETTE["dark"]
            )

        # Lost revenue bar overlay on current
        lost = (self.lost_monthly_rev) / 1e5
        ax.bar(
            labels[0], lost,
            bottom=revenues[0],
            color=PALETTE["red"], alpha=0.75, width=0.55, zorder=3,
            label=f"Lost to no-shows: ₹{lost:.1f}L"
        )
        ax.text(
            0, revenues[0] + lost / 2,
            f"-₹{lost:.1f}L", ha="center", va="center",
            fontsize=9, color="white", fontweight="bold"
        )

        ax.axhline(self.max_monthly_rev / 1e5, color=PALETTE["teal"],
                   linestyle="--", linewidth=1.2, alpha=0.5, label="Max potential")
        ax.set_ylabel("Monthly Revenue (₹ Lakh)", fontsize=11)
        ax.set_title("Revenue Gap Analysis — Current State vs. Targets",
                     fontsize=13, fontweight="bold", pad=12)
        ax.legend(fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)
        ax.set_ylim(0, self.max_monthly_rev / 1e5 * 1.18)
        ax.spines[["top", "right"]].set_visible(False)
        return ax


# MODULE 2: NO-SHOW PREDICTION MODEL

In [8]:
class NoShowPredictionModel:
    """
    Trains a no-show prediction classifier on synthetic appointment data.

    Features used (mirrors what the report specifies):
      - patient_age_group       : 0=18-30, 1=31-50, 2=51+
      - appointment_lead_days   : days between booking and appointment
      - day_of_week             : 0=Monday … 4=Friday
      - time_slot               : 0=morning, 1=afternoon, 2=evening
      - past_no_show_count      : historical no-shows for this patient
      - distance_km             : patient distance from clinic
      - appointment_type        : 0=follow-up, 1=new, 2=specialist
      - clinic_id               : 0–4

    Output: probability score 0–1 per appointment.
    Risk bands:
      < 0.30  → Low  (1 reminder)
      0.30–0.60 → Medium (2 reminders)
      > 0.60  → High (3 reminders + mandatory confirm)
    """

    def __init__(self, n_samples: int = 5000):
        self.n_samples = n_samples
        self.scaler    = StandardScaler()
        self.model     = GradientBoostingClassifier(
            n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42
        )
        self.feature_names = [
            "patient_age_group", "appointment_lead_days", "day_of_week",
            "time_slot", "past_no_show_count", "distance_km",
            "appointment_type", "clinic_id",
        ]
        self.X_train = self.X_test = self.y_train = self.y_test = None
        self._generate_data()

    def _generate_data(self):
        """
        Synthetic dataset designed so that longer lead times, more past
        no-shows, greater distance, and evening slots raise no-show probability.
        """
        n = self.n_samples
        age_group   = np.random.choice([0, 1, 2], n, p=[0.30, 0.45, 0.25])
        lead_days   = np.random.randint(1, 30, n)
        day_of_week = np.random.randint(0, 5, n)
        time_slot   = np.random.choice([0, 1, 2], n, p=[0.40, 0.35, 0.25])
        past_ns     = np.random.poisson(1.2, n).clip(0, 8)
        distance    = np.abs(np.random.normal(8, 5, n)).clip(0.5, 40)
        appt_type   = np.random.choice([0, 1, 2], n, p=[0.50, 0.35, 0.15])
        clinic_id   = np.random.randint(0, 5, n)

        # Logistic probability construction (interpretable)
        log_odds = (
            -1.5
            + 0.04  * lead_days
            + 0.35  * (time_slot == 2).astype(int)
            + 0.45  * past_ns
            + 0.03  * distance
            - 0.20  * (age_group == 2).astype(int)
            + 0.25  * (appt_type == 1).astype(int)
            - 0.10  * (day_of_week == 0).astype(int)
        )
        prob  = 1 / (1 + np.exp(-log_odds))
        label = (np.random.random(n) < prob).astype(int)

        self.df = pd.DataFrame({
            "patient_age_group":       age_group,
            "appointment_lead_days":   lead_days,
            "day_of_week":             day_of_week,
            "time_slot":               time_slot,
            "past_no_show_count":      past_ns,
            "distance_km":             distance.round(1),
            "appointment_type":        appt_type,
            "clinic_id":               clinic_id,
            "no_show":                 label,
        })

    def train(self):
        X = self.df[self.feature_names].values
        y = self.df["no_show"].values
        X_s = self.scaler.fit_transform(X)
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X_s, y, test_size=0.25, random_state=42, stratify=y
        )
        self.model.fit(self.X_train, self.y_train)
        return self

    def evaluate(self) -> dict:
        y_pred  = self.model.predict(self.X_test)
        y_prob  = self.model.predict_proba(self.X_test)[:, 1]
        auc     = roc_auc_score(self.y_test, y_prob)
        report  = classification_report(self.y_test, y_pred, output_dict=True)
        return {"auc": auc, "report": report, "y_prob": y_prob, "y_test": self.y_test}

    def predict_risk(self, appointment_features: np.ndarray) -> np.ndarray:
        """Return probability scores for new appointments."""
        scaled = self.scaler.transform(appointment_features)
        return self.model.predict_proba(scaled)[:, 1]

    def assign_risk_band(self, score: float) -> str:
        if score < 0.30:
            return "Low"
        elif score < 0.60:
            return "Medium"
        else:
            return "High"

    def reminder_protocol(self, score: float) -> dict:
        band = self.assign_risk_band(score)
        protocols = {
            "Low":    {"reminders": 1, "touchpoints": ["24h before"],
                       "channel": "WhatsApp", "confirm_required": False},
            "Medium": {"reminders": 2, "touchpoints": ["48h before", "2h before"],
                       "channel": "WhatsApp + SMS", "confirm_required": False},
            "High":   {"reminders": 3, "touchpoints": ["48h", "24h", "2h before"],
                       "channel": "WhatsApp + SMS + Call", "confirm_required": True},
        }
        return {"risk_band": band, "score": round(score, 3), **protocols[band]}

    def plot_feature_importance(self, ax=None):
        importance = self.model.feature_importances_
        idx = np.argsort(importance)
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 4))
        colors = [PALETTE["teal"] if i == idx[-1] else PALETTE["blue"] for i in idx]
        ax.barh(
            [self.feature_names[i].replace("_", " ").title() for i in idx],
            importance[idx], color=colors, zorder=3
        )
        ax.set_title("Feature Importance — No-Show Prediction Model",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Importance Score")
        ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

    def plot_risk_distribution(self, ax=None):
        scores = self.model.predict_proba(self.X_test)[:, 1]
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 4))
        bins = np.linspace(0, 1, 35)
        ax.hist(scores[self.y_test == 0], bins=bins, alpha=0.7,
                color=PALETTE["teal"], label="Attended", zorder=3)
        ax.hist(scores[self.y_test == 1], bins=bins, alpha=0.7,
                color=PALETTE["red"], label="No-Show", zorder=3)
        ax.axvline(0.30, color=PALETTE["amber"], linestyle="--",
                   linewidth=1.5, label="Low/Med threshold (0.30)")
        ax.axvline(0.60, color=PALETTE["purple"], linestyle="--",
                   linewidth=1.5, label="Med/High threshold (0.60)")
        ax.set_title("No-Show Risk Score Distribution",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Predicted No-Show Probability")
        ax.set_ylabel("Patient Count")
        ax.legend(fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

# MODULE 3: SMART OVERBOOKING ENGINE

In [9]:
class SmartOverbookingEngine:
    """
    Calculates optimal overbooking buffers per time slot and clinic
    based on predicted no-show rates, with a conservative starting cap.

    Strategy:
      buffer_count = floor(slot_capacity × predicted_no_show_rate)
      Starting cap: max 10% of slot capacity until validation is complete.
    """

    def __init__(self, slot_capacity: int = 8, initial_cap: float = 0.10):
        self.slot_capacity = slot_capacity
        self.initial_cap   = initial_cap
        self.time_bands    = ["08:00–10:00", "10:00–12:00", "12:00–14:00",
                               "14:00–16:00", "16:00–18:00"]
        # Historical no-show rates per time band (based on report assumptions)
        self.historical_ns = {
            "08:00–10:00": 0.22,
            "10:00–12:00": 0.25,
            "12:00–14:00": 0.35,
            "14:00–16:00": 0.28,
            "16:00–18:00": 0.38,
        }

    def compute_overbooking_plan(self) -> pd.DataFrame:
        rows = []
        for band, ns_rate in self.historical_ns.items():
            raw_buffer  = self.slot_capacity * ns_rate
            cap_buffer  = min(raw_buffer, self.slot_capacity * self.initial_cap)
            safe_buffer = max(1, round(cap_buffer))
            rows.append({
                "Time band":              band,
                "Historical no-show %":  f"{ns_rate:.0%}",
                "Raw buffer (slots)":    round(raw_buffer, 1),
                "Conservative buffer":   safe_buffer,
                "Total booked":          self.slot_capacity + safe_buffer,
                "Expected attended":     round(self.slot_capacity * (1 - ns_rate) + safe_buffer * 0.5),
                "Revenue uplift/day (₹)":safe_buffer * REVENUE_PER_VISIT,
            })
        df = pd.DataFrame(rows)
        df["Monthly uplift (₹)"] = df["Revenue uplift/day (₹)"] * WORKING_DAYS * CLINICS
        return df

    def simulate_waitlist_fill(self, days: int = 26, fill_rate: float = 0.55) -> pd.DataFrame:
        """
        Monte-Carlo simulation: how many waitlisted patients accept same-day slots?
        fill_rate = probability a waitlisted patient accepts within 30-min window.
        """
        results = []
        for day in range(1, days + 1):
            daily_slots_opened = 0
            daily_filled       = 0
            for band, ns_rate in self.historical_ns.items():
                slots_opened = np.random.binomial(self.slot_capacity, ns_rate)
                filled       = np.random.binomial(slots_opened, fill_rate)
                daily_slots_opened += slots_opened
                daily_filled       += filled
            results.append({
                "Day": day,
                "Slots opened": daily_slots_opened,
                "Waitlist filled": daily_filled,
                "Fill rate": daily_filled / max(daily_slots_opened, 1),
                "Added revenue (₹)": daily_filled * REVENUE_PER_VISIT,
            })
        return pd.DataFrame(results)

    def plot_overbooking_plan(self, ax=None):
        plan = self.compute_overbooking_plan()
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 4))
        x     = np.arange(len(self.time_bands))
        width = 0.3
        ax.bar(x - width/2, [self.slot_capacity]*5, width, color=PALETTE["gray"],
               alpha=0.7, label="Base slots", zorder=3)
        ax.bar(x + width/2, plan["Conservative buffer"], width, color=PALETTE["teal"],
               alpha=0.9, label="Overbooking buffer (10% cap)", zorder=3)
        for i, (_, row) in enumerate(plan.iterrows()):
            ax.text(i + width/2, row["Conservative buffer"] + 0.05,
                    f'+{int(row["Conservative buffer"])}',
                    ha="center", va="bottom", fontsize=9,
                    color=PALETTE["teal"], fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(self.time_bands, rotation=15, ha="right", fontsize=9)
        ax.set_ylabel("Slots")
        ax.set_title("Smart Overbooking Buffer by Time Band",
                     fontsize=12, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

    def plot_waitlist_simulation(self, ax=None):
        sim = self.simulate_waitlist_fill()
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 4))
        ax2 = ax.twinx()
        ax.fill_between(sim["Day"], sim["Added revenue (₹)"] / 1000,
                        alpha=0.25, color=PALETTE["teal"])
        ax.plot(sim["Day"], sim["Added revenue (₹)"] / 1000,
                color=PALETTE["teal"], linewidth=2, label="Added revenue (₹K/day)")
        ax2.plot(sim["Day"], sim["Fill rate"] * 100,
                 color=PALETTE["amber"], linestyle="--",
                 linewidth=1.8, label="Waitlist fill rate (%)")
        ax.set_xlabel("Day of Month")
        ax.set_ylabel("Added Revenue (₹ Thousands)", color=PALETTE["teal"])
        ax2.set_ylabel("Fill Rate (%)", color=PALETTE["amber"])
        ax.set_title("Waitlist Fill Simulation — Monthly View",
                     fontsize=12, fontweight="bold")
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc="lower right")
        ax.grid(linestyle="--", alpha=0.3, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

# MODULE 4: PATIENT SEGMENTATION (RFM)

In [10]:
class PatientSegmentation:
    """
    RFM (Recency, Frequency, Monetary) segmentation across the patient base.

    Segments:
      Champions     — high F, high M, recent R
      At-Risk       — previously regular, no recent visit
      High No-Show  — multiple historical no-shows
      New Patients  — 1–2 visits only

    Engagement strategy per segment is defined in the report.
    """

    SEGMENT_STRATEGIES = {
        "Champions": {
            "reminders": 1,
            "message": "Loyalty nudge + health tip + thank-you",
            "channel": "WhatsApp",
            "goal": "Retention & LTV growth",
            "color": PALETTE["teal"],
        },
        "At-Risk": {
            "reminders": 2,
            "message": "'We miss you' + 1-tap rebooking link",
            "channel": "WhatsApp + SMS",
            "goal": "Reactivation",
            "color": PALETTE["amber"],
        },
        "High No-Show": {
            "reminders": 3,
            "message": "Extra reminders + mandatory 2h confirm",
            "channel": "WhatsApp + SMS + Call",
            "goal": "Reduce no-show rate",
            "color": PALETTE["red"],
        },
        "New Patients": {
            "reminders": 2,
            "message": "Trust-building onboarding (3-msg sequence)",
            "channel": "WhatsApp",
            "goal": "First-visit conversion",
            "color": PALETTE["blue"],
        },
    }

    def __init__(self, n_patients: int = 1200):
        self.n_patients = n_patients
        self._generate_patient_base()

    def _generate_patient_base(self):
        n = self.n_patients
        recency   = np.random.randint(1, 365, n)   # days since last visit
        frequency = np.random.poisson(5, n).clip(1, 24)
        monetary  = frequency * REVENUE_PER_VISIT * np.random.uniform(0.8, 1.2, n)
        no_shows  = np.random.poisson(1.0, n).clip(0, 8)

        self.patients = pd.DataFrame({
            "patient_id": [f"P{str(i+1).zfill(5)}" for i in range(n)],
            "recency_days":   recency,
            "visit_frequency":frequency,
            "total_spend":    monetary.round(0),
            "no_show_count":  no_shows,
        })
        self._assign_segments()

    def _score_rfm(self, df: pd.DataFrame) -> pd.DataFrame:
        df["R_score"] = pd.qcut(df["recency_days"].rank(method="first"),
                                 4, labels=[4, 3, 2, 1]).astype(int)
        df["F_score"] = pd.qcut(df["visit_frequency"].rank(method="first"),
                                 4, labels=[1, 2, 3, 4]).astype(int)
        df["M_score"] = pd.qcut(df["total_spend"].rank(method="first"),
                                 4, labels=[1, 2, 3, 4]).astype(int)
        df["RFM_total"] = df["R_score"] + df["F_score"] + df["M_score"]
        return df

    def _assign_segments(self):
        df = self._score_rfm(self.patients.copy())

        conditions = [
            (df["no_show_count"] >= 3),
            (df["visit_frequency"] <= 2),
            (df["RFM_total"] >= 9),
            (df["RFM_total"] <= 5) | (df["recency_days"] > 180),
        ]
        choices = ["High No-Show", "New Patients", "Champions", "At-Risk"]
        df["segment"] = np.select(conditions, choices, default="At-Risk")
        self.patients = df

    def segment_summary(self) -> pd.DataFrame:
        summary = (
            self.patients.groupby("segment")
            .agg(
                count          =("patient_id", "count"),
                avg_frequency  =("visit_frequency", "mean"),
                avg_spend      =("total_spend", "mean"),
                avg_no_shows   =("no_show_count", "mean"),
                avg_recency    =("recency_days", "mean"),
            )
            .round(1)
            .reset_index()
        )
        summary["share_%"] = (summary["count"] / summary["count"].sum() * 100).round(1)
        summary["engagement_strategy"] = summary["segment"].map(
            lambda s: self.SEGMENT_STRATEGIES[s]["message"]
        )
        return summary

    def plot_segment_distribution(self, ax=None):
        summary = self.segment_summary()
        segments = summary["segment"].tolist()
        counts   = summary["count"].tolist()
        colors   = [self.SEGMENT_STRATEGIES[s]["color"] for s in segments]

        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 6))

        wedges, texts, autotexts = ax.pie(
            counts, labels=segments, autopct="%1.1f%%",
            colors=colors, startangle=140,
            wedgeprops={"edgecolor": "white", "linewidth": 2},
            textprops={"fontsize": 10},
        )
        for at in autotexts:
            at.set_fontsize(9)
            at.set_fontweight("bold")
            at.set_color("white")
        ax.set_title("Patient Segmentation — RFM Model",
                     fontsize=12, fontweight="bold", pad=14)
        return ax

    def plot_rfm_scatter(self, ax=None):
        df = self.patients.sample(400, random_state=42)
        segment_colors = {
            s: self.SEGMENT_STRATEGIES[s]["color"] for s in self.SEGMENT_STRATEGIES
        }
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 5))
        for seg, grp in df.groupby("segment"):
            ax.scatter(
                grp["visit_frequency"], grp["recency_days"],
                c=segment_colors[seg], label=seg, alpha=0.6, s=35, edgecolors="none"
            )
        ax.set_xlabel("Visit Frequency", fontsize=10)
        ax.set_ylabel("Recency (days since last visit)", fontsize=10)
        ax.set_title("Patient Segments — Frequency vs. Recency",
                     fontsize=12, fontweight="bold")
        ax.legend(fontsize=9)
        ax.invert_yaxis()
        ax.grid(linestyle="--", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

# MODULE 5: AUTOMATED REMINDER SCHEDULER

In [11]:
class ReminderScheduler:
    """
    Simulates the multi-channel automated reminder workflow.
    Given a list of appointments with risk scores, generates the full
    reminder schedule per patient with channel selection.
    """

    CHANNELS = {
        "WhatsApp": {"delivery_rate": 0.92, "cost_per_msg": 0.50},
        "SMS":      {"delivery_rate": 0.97, "cost_per_msg": 0.30},
        "Call":     {"delivery_rate": 0.85, "cost_per_msg": 2.00},
    }

    def generate_daily_schedule(
        self,
        model: NoShowPredictionModel,
        n_appointments: int = TOTAL_SLOTS_DAY
    ) -> pd.DataFrame:
        """
        For a full day of appointments, assign risk bands and reminder schedules.
        """
        # Synthetic appointment features for one day
        features = np.column_stack([
            np.random.choice([0, 1, 2], n_appointments),
            np.random.randint(1, 20, n_appointments),
            np.random.randint(0, 5, n_appointments),
            np.random.choice([0, 1, 2], n_appointments),
            np.random.poisson(1.2, n_appointments).clip(0, 8),
            np.abs(np.random.normal(8, 5, n_appointments)).clip(0.5, 40),
            np.random.choice([0, 1, 2], n_appointments),
            np.random.randint(0, 5, n_appointments),
        ])
        scores = model.predict_risk(features)
        bands  = [model.assign_risk_band(s) for s in scores]

        schedule = []
        for i, (score, band) in enumerate(zip(scores, bands)):
            protocol = model.reminder_protocol(score)
            schedule.append({
                "appointment_id": f"APT{str(i+1).zfill(4)}",
                "risk_score":     round(score, 3),
                "risk_band":      band,
                "reminders":      protocol["reminders"],
                "channel":        protocol["channel"],
                "confirm_req":    protocol["confirm_required"],
                "touchpoints":    " → ".join(protocol["touchpoints"]),
            })
        return pd.DataFrame(schedule)

    def compute_daily_cost(self, schedule: pd.DataFrame) -> dict:
        total_msgs = schedule["reminders"].sum()
        # Weighted channel cost
        band_counts = schedule["risk_band"].value_counts()
        cost = (
            band_counts.get("Low", 0) * 1 * self.CHANNELS["WhatsApp"]["cost_per_msg"]
            + band_counts.get("Medium", 0) * 2 * (
                self.CHANNELS["WhatsApp"]["cost_per_msg"] + self.CHANNELS["SMS"]["cost_per_msg"]
            ) / 2
            + band_counts.get("High", 0) * 3 * (
                self.CHANNELS["WhatsApp"]["cost_per_msg"]
                + self.CHANNELS["SMS"]["cost_per_msg"]
                + self.CHANNELS["Call"]["cost_per_msg"]
            ) / 3
        )
        return {
            "total_messages_sent": int(total_msgs),
            "estimated_cost_inr":  round(cost, 2),
            "monthly_cost_inr":    round(cost * WORKING_DAYS, 2),
            "band_counts":         band_counts.to_dict(),
        }

    def plot_reminder_band_breakdown(self, schedule: pd.DataFrame, ax=None):
        bands  = ["Low", "Medium", "High"]
        counts = [schedule[schedule["risk_band"] == b].shape[0] for b in bands]
        colors = [PALETTE["teal"], PALETTE["amber"], PALETTE["red"]]

        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 4))
        bars = ax.bar(bands, counts, color=colors, width=0.45, zorder=3)
        for bar, count in zip(bars, counts):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1.5,
                f"{count}\npatients",
                ha="center", va="bottom", fontsize=10, fontweight="bold",
                color=PALETTE["dark"]
            )
        protocols_text = {
            "Low":    "1 reminder\n(WhatsApp)",
            "Medium": "2 reminders\n(WA + SMS)",
            "High":   "3 reminders\n+ Confirm",
        }
        for i, (band, txt) in enumerate(protocols_text.items()):
            ax.text(i, -12, txt, ha="center", va="top", fontsize=8,
                    color=colors[i], style="italic")
        ax.set_title("Daily Reminder Schedule — Risk Band Breakdown",
                     fontsize=12, fontweight="bold")
        ax.set_ylabel("Appointments")
        ax.set_ylim(-20, max(counts) * 1.3)
        ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

# MODULE 6: KPI DASHBOARD & PROJECTIONS

In [12]:
class KPIDashboard:
    """
    Projects KPI trajectories across 12 months based on phased interventions.
    Mirrors the report's Figure 8 (Implementation Roadmap) and Figure 13
    (Projected Outcomes Summary).
    """

    MONTHS = ["Baseline", "M1", "M2", "M3", "M4", "M5",
              "M6", "M7", "M8", "M9", "M10", "M11", "M12"]

    # No-show rate projection (declining S-curve as interventions compound)
    NO_SHOW_TRAJECTORY = [0.30, 0.28, 0.26, 0.22, 0.20, 0.19,
                          0.18, 0.17, 0.165, 0.16, 0.155, 0.15, 0.14]

    # Slot utilisation mirrors no-show reduction
    UTILISATION_TRAJECTORY = [1 - r for r in NO_SHOW_TRAJECTORY]

    def revenue_trajectory(self) -> list:
        return [
            round(TOTAL_SLOTS_DAY * (1 - ns) * REVENUE_PER_VISIT * WORKING_DAYS / 1e5, 2)
            for ns in self.NO_SHOW_TRAJECTORY
        ]

    def patients_per_day_trajectory(self) -> list:
        return [round(TOTAL_SLOTS_DAY * (1 - ns)) for ns in self.NO_SHOW_TRAJECTORY]

    def uplift_trajectory(self) -> list:
        base = self.revenue_trajectory()[0]
        return [round(r - base, 2) for r in self.revenue_trajectory()]

    def summary_table(self) -> pd.DataFrame:
        return pd.DataFrame({
            "Month":             self.MONTHS,
            "No-show rate":      [f"{r:.0%}" for r in self.NO_SHOW_TRAJECTORY],
            "Patients/day":      self.patients_per_day_trajectory(),
            "Monthly rev (₹L)":  self.revenue_trajectory(),
            "Uplift vs base (₹L)": self.uplift_trajectory(),
            "Utilisation":       [f"{u:.0%}" for u in self.UTILISATION_TRAJECTORY],
        })

    def plot_kpi_projection(self, ax=None):
        revs   = self.revenue_trajectory()
        ns     = [r * 100 for r in self.NO_SHOW_TRAJECTORY]
        util   = [u * 100 for u in self.UTILISATION_TRAJECTORY]
        months = range(len(self.MONTHS))

        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 5))
        ax2 = ax.twinx()

        ax.fill_between(months, revs, alpha=0.12, color=PALETTE["teal"])
        ax.plot(months, revs, color=PALETTE["teal"], linewidth=2.5,
                marker="o", markersize=5, label="Monthly revenue (₹L)")

        ax2.plot(months, ns, color=PALETTE["red"], linestyle="--",
                 linewidth=2, marker="s", markersize=4, label="No-show rate (%)")
        ax2.plot(months, util, color=PALETTE["blue"], linestyle=":",
                 linewidth=1.8, marker="^", markersize=4, label="Utilisation (%)")

        # Phase shading
        ax.axvspan(1, 3, alpha=0.06, color=PALETTE["blue"],
                   label="Phase 1: Quick Wins")
        ax.axvspan(3, 12, alpha=0.06, color=PALETTE["teal"],
                   label="Phase 2: Scale & Intelligence")
        ax.axvline(3, color=PALETTE["blue"], linestyle="-.", linewidth=1, alpha=0.5)

        # Annotations
        ax.annotate("Phase 1\nKPI check", xy=(3, revs[3]),
                    xytext=(3.4, revs[3] - 1.5),
                    fontsize=8, color=PALETTE["blue"],
                    arrowprops=dict(arrowstyle="->", color=PALETTE["blue"], lw=1))
        ax.annotate("Phase 2\nKPI check", xy=(12, revs[12]),
                    xytext=(10.5, revs[12] - 1.8),
                    fontsize=8, color=PALETTE["teal"],
                    arrowprops=dict(arrowstyle="->", color=PALETTE["teal"], lw=1))

        ax.set_xticks(months)
        ax.set_xticklabels(self.MONTHS, fontsize=9)
        ax.set_ylabel("Monthly Revenue (₹ Lakh)", fontsize=10, color=PALETTE["teal"])
        ax2.set_ylabel("Rate / Utilisation (%)", fontsize=10)
        ax.set_title("12-Month KPI Projection — Revenue & No-Show Rate",
                     fontsize=13, fontweight="bold")

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8,
                  loc="upper left", ncol=2)
        ax.grid(linestyle="--", alpha=0.25, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        return ax

    def plot_kpi_scorecard(self, ax=None):
        """Visual scorecard for 3-month and 12-month checkpoints."""
        kpis = [
            ("No-show rate",           "30%",  "≤ 22%",  "≤ 14%"),
            ("Daily patients",         "140",  "155",    "172"),
            ("Monthly revenue",        "₹25.5L","₹29–30L","₹32–34L"),
            ("Reminder response rate", "0%",   "≥ 60%",  "≥ 70%"),
            ("Reschedule rate",        "~0%",  "≥ 25%",  "≥ 40%"),
            ("Slot utilisation",       "70%",  "77%",    "86%"),
        ]
        col_labels = ["KPI", "Baseline", "3-Month Target", "12-Month Target"]
        cell_text  = [[k[0], k[1], k[2], k[3]] for k in kpis]

        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 3))
        ax.axis("off")
        table = ax.table(
            cellText   = cell_text,
            colLabels  = col_labels,
            cellLoc    = "center",
            loc        = "center",
        )
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.6)

        # Header row
        for j in range(4):
            table[(0, j)].set_facecolor(PALETTE["dark"])
            table[(0, j)].set_text_props(color="white", fontweight="bold")
        # Baseline column
        for i in range(1, len(kpis) + 1):
            table[(i, 1)].set_facecolor("#F1EFE8")
        # 3-month column
        for i in range(1, len(kpis) + 1):
            table[(i, 2)].set_facecolor("#E6F1FB")
            table[(i, 2)].set_text_props(color=PALETTE["blue"])
        # 12-month column
        for i in range(1, len(kpis) + 1):
            table[(i, 3)].set_facecolor("#E1F5EE")
            table[(i, 3)].set_text_props(color=PALETTE["teal"])
        # Alternating row shading
        for i in range(1, len(kpis) + 1):
            if i % 2 == 0:
                table[(i, 0)].set_facecolor("#FAFAF8")

        ax.set_title("KPI Scorecard — Baseline vs. Phase Targets",
                     fontsize=12, fontweight="bold", pad=10)
        return ax

# MASTER DASHBOARD: Generate all figures into one output file

In [13]:
def generate_master_report():
    print("=" * 65)
    print("  HEALTHCARE CLINIC OPTIMIZATION — STRATEGY SIMULATION")
    print("=" * 65)

    # ── Instantiate all modules ──────────────────────────────────────
    print("\n[1/6] Current State Analysis ...")
    csa   = CurrentStateAnalysis()
    print(csa.summary().to_string(index=False))

    print("\n[2/6] Training No-Show Prediction Model ...")
    nsp   = NoShowPredictionModel(n_samples=6000)
    nsp.train()
    eval_metrics = nsp.evaluate()
    print(f"      ROC-AUC Score      : {eval_metrics['auc']:.4f}")
    print(f"      No-show Precision  : {eval_metrics['report']['1']['precision']:.3f}")
    print(f"      No-show Recall     : {eval_metrics['report']['1']['recall']:.3f}")
    print(f"      No-show F1         : {eval_metrics['report']['1']['f1-score']:.3f}")

    # Sample reminder protocols
    test_scores = [0.18, 0.44, 0.72]
    print("\n      Sample Reminder Protocols:")
    for score in test_scores:
        p = nsp.reminder_protocol(score)
        print(f"        Score {score:.2f} → {p['risk_band']:6s} | "
              f"{p['reminders']} reminder(s) | "
              f"Channel: {p['channel']} | "
              f"Confirm required: {p['confirm_required']}")

    print("\n[3/6] Smart Overbooking Engine ...")
    obe   = SmartOverbookingEngine()
    plan  = obe.compute_overbooking_plan()
    total_monthly_uplift = plan["Monthly uplift (₹)"].sum()
    print(plan[["Time band", "Historical no-show %",
                "Conservative buffer", "Monthly uplift (₹)"]].to_string(index=False))
    print(f"\n      Total monthly revenue uplift from overbooking: "
          f"₹{total_monthly_uplift:,.0f}")

    print("\n[4/6] Patient Segmentation (RFM) ...")
    ps    = PatientSegmentation(n_patients=1500)
    seg_summary = ps.segment_summary()
    print(seg_summary[["segment", "count", "share_%",
                        "avg_frequency", "avg_no_shows"]].to_string(index=False))

    print("\n[5/6] Reminder Scheduler — Daily Schedule ...")
    sched_obj = ReminderScheduler()
    daily_sched = sched_obj.generate_daily_schedule(nsp)
    cost_info   = sched_obj.compute_daily_cost(daily_sched)
    print(f"      Daily messages sent  : {cost_info['total_messages_sent']}")
    print(f"      Band distribution    : {cost_info['band_counts']}")
    print(f"      Daily cost (₹)       : ₹{cost_info['estimated_cost_inr']}")
    print(f"      Monthly cost (₹)     : ₹{cost_info['monthly_cost_inr']}")

    print("\n[6/6] Generating Master Dashboard ...")
    kpi   = KPIDashboard()
    print("\n      12-Month KPI Projection:")
    print(kpi.summary_table()[
        ["Month", "No-show rate", "Monthly rev (₹L)", "Uplift vs base (₹L)"]
    ].to_string(index=False))

    # ── Build Master Figure ──────────────────────────────────────────
    fig = plt.figure(figsize=(20, 26))
    fig.patch.set_facecolor("white")
    gs  = gridspec.GridSpec(5, 2, figure=fig,
                            hspace=0.52, wspace=0.38,
                            left=0.07, right=0.96,
                            top=0.94, bottom=0.04)

    # Title
    fig.suptitle(
        "Healthcare Clinic Chain — AI Optimization Strategy\nSimulation Report",
        fontsize=18, fontweight="bold", y=0.97, color=PALETTE["dark"]
    )

    # Row 0: Revenue gap + KPI projection
    ax0 = fig.add_subplot(gs[0, 0])
    csa.plot_revenue_gap(ax0)

    ax1 = fig.add_subplot(gs[0, 1])
    kpi.plot_kpi_projection(ax1)

    # Row 1: Feature importance + risk distribution
    ax2 = fig.add_subplot(gs[1, 0])
    nsp.plot_feature_importance(ax2)

    ax3 = fig.add_subplot(gs[1, 1])
    nsp.plot_risk_distribution(ax3)

    # Row 2: Overbooking plan + waitlist simulation
    ax4 = fig.add_subplot(gs[2, 0])
    obe.plot_overbooking_plan(ax4)

    ax5 = fig.add_subplot(gs[2, 1])
    obe.plot_waitlist_simulation(ax5)

    # Row 3: Segmentation pie + RFM scatter
    ax6 = fig.add_subplot(gs[3, 0])
    ps.plot_segment_distribution(ax6)

    ax7 = fig.add_subplot(gs[3, 1])
    ps.plot_rfm_scatter(ax7)

    # Row 4: Reminder band breakdown + KPI scorecard
    ax8 = fig.add_subplot(gs[4, 0])
    sched_obj.plot_reminder_band_breakdown(daily_sched, ax8)

    ax9 = fig.add_subplot(gs[4, 1])
    kpi.plot_kpi_scorecard(ax9)

    # Footer
    fig.text(
        0.5, 0.01,
        "All projections are model estimates based on stated operating parameters and India outpatient healthcare benchmarks. "
        "Simulation uses synthetic data representative of clinic appointment patterns.",
        ha="center", fontsize=8, color=PALETTE["gray"], style="italic"
    )

    out_path = os.path.join(OUTPUT_DIR, "clinic_optimization_dashboard.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"\n  ✓ Dashboard saved → {out_path}")
    plt.close(fig)

    print("\n" + "=" * 65)
    print("  SIMULATION COMPLETE")
    print("  Modules run: 6 / 6")
    print(f"  Projected monthly revenue uplift (Month 12): "
          f"₹{(kpi.revenue_trajectory()[12] - kpi.revenue_trajectory()[0]):.2f}L")
    print(f"  No-show rate reduction: "
          f"{kpi.NO_SHOW_TRAJECTORY[0]:.0%} → {kpi.NO_SHOW_TRAJECTORY[12]:.0%}")
    print(f"  Effective patient throughput: "
          f"{kpi.patients_per_day_trajectory()[0]}/day → "
          f"{kpi.patients_per_day_trajectory()[12]}/day")
    print("=" * 65)

    return {
        "current_state":    csa,
        "prediction_model": nsp,
        "overbooking":      obe,
        "segmentation":     ps,
        "scheduler":        sched_obj,
        "kpi_dashboard":    kpi,
    }

# ENTRY POINT

In [14]:
if __name__ == "__main__":
    modules = generate_master_report()


  HEALTHCARE CLINIC OPTIMIZATION — STRATEGY SIMULATION

[1/6] Current State Analysis ...
                                    Metric      Value
          Total scheduled appointments/day        200
Actual patients seen/day (70% utilisation)        140
                         No-show slots/day         60
                  Actual daily revenue (₹)    ₹98,000
                Actual monthly revenue (₹) ₹2,548,000
         Theoretical max daily revenue (₹)   ₹140,000
       Theoretical max monthly revenue (₹) ₹3,640,000
        Daily revenue LOST to no-shows (₹)    ₹42,000
      Monthly revenue LOST to no-shows (₹) ₹1,092,000
             Current slot utilisation rate        70%

[2/6] Training No-Show Prediction Model ...
      ROC-AUC Score      : 0.6380
      No-show Precision  : 0.605
      No-show Recall     : 0.525
      No-show F1         : 0.562

      Sample Reminder Protocols:
        Score 0.18 → Low    | 1 reminder(s) | Channel: WhatsApp | Confirm required: False
        Score 0